In [0]:
from pyspark.sql.functions import col, lower, upper, trim, when, current_timestamp, coalesce, try_to_date, round, lit

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

print(" Silver schema ready")

In [0]:
customers = (
    spark.table("azure_blob_storage.src_customers")
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("customer_name", coalesce(lower(trim(col("customer_name"))), lit("none")))
    .withColumn("customer_type", coalesce(lower(trim(col("customer_type"))), lit("none")))
    .withColumn("country", 
        when(upper(trim(col("country"))).isin(["USA", "U.S.A", "U.S.A.", "UNITED STATES"]), "USA")
        .when(upper(trim(col("country"))) == "INDIA", "INDIA")
        .otherwise(coalesce(upper(trim(col("country"))), lit("NONE")))
    )
    .withColumn("ingestion_ts", current_timestamp())
    .dropDuplicates(["customer_id"])
)

customers.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.customers")
print(f"customers: {customers.count()} records")
display(customers)